In [ ]:
#| default_exp rag

In [ ]:
#| hide
from nbdev.showdoc import *

## LiteRAG

`LiteRAG` combines litesearch's hybrid FTS5+vector search with [Lisette](https://github.com/AnswerDotAI/lisette) (LiteLLM wrapper) and an optional `FastRerank` cross-encoder into a single agentic RAG interface.

- **No `@tool` decorators** — tools are plain Python closures injected via Lisette's `tools=` parameter.
- **Single `.db` file** — everything lives in the existing `store` table.
- **LiteLLM-compatible** — works with any `llm_model` string: `'ollama/qwen2.5'`, `'claude-haiku-4-5-20251001'`, etc.

In [ ]:
#| export
from fastcore.all import ifnone, store_attr, L
from litesearch.core import database
from typing import Optional
import json

In [ ]:
#| export
_RAG_SYSTEM_PROMPT = """You are a research assistant with access to a local document corpus.

Steps:
1. ALWAYS call search_knowledge_base first with the user's query.
2. Cite retrieved passages by number [1], [2], etc.
3. Only use web search if the local corpus lacks sufficient information.
4. If you need more detail, call search_knowledge_base again with a refined query.
5. Be concise and precise. Do not fabricate citations."""

In [ ]:
#| export
class LiteRAG:
	def __init__(
		self,
		db_path: str = ':memory:',                       # path to SQLite DB or ':memory:'
		encoder_model: str = 'minishlab/potion-retrieval-32M', # HF model for chonkie AutoEmbeddings
		llm_model: str = 'ollama/qwen2.5',               # any LiteLLM model string
		search_level: Optional[str] = 'm',               # Lisette web search: 'l'|'m'|'h'|None
		k: int = 10,                                      # candidate docs before rerank
		top_n: int = 3,                                   # final passages passed to LLM
		expand_parents: bool = True,                     # fetch parent chunks for child hits
		reranker=None,                                    # FastRerank instance or None
		tools: Optional[list] = None,                    # additional plain-function Lisette tools
	):
		"""RAG + agentic query over a litesearch SQLite store.
		Uses Chonkie AutoEmbeddings for retrieval and Lisette Chat for generation.
		reranker: FastRerank instance (optional, improves precision).
		tools: extra plain Python functions — no @tool decorators needed."""
		store_attr()
		self.db = database(db_path)
		self.store = self.db.get_store()
		from chonkie import AutoEmbeddings
		_enc = AutoEmbeddings().get_embeddings(encoder_model)
		self._emb_fn = _enc.embed

	def retrieve(self, query: str) -> list:
		"""Hybrid FTS5+vector RRF search → optional parent expansion → optional rerank."""
		from litesearch.data import pre
		emb = self._emb_fn([query])[0].tobytes()
		results = list(self.db.search(pre(query), emb, limit=self.k) or [])
		if self.expand_parents and results:
			parent_ids = {r.get('parent_id') for r in results if r.get('parent_id')}
			if parent_ids:
				ph = ','.join('?' * len(parent_ids))
				parents = list(self.db.q(
					f'SELECT * FROM store WHERE id IN ({ph})', list(parent_ids)))
				seen = {r.get('rowid') or r.get('id') for r in results}
				results += [p for p in parents
							if (p.get('rowid') or p.get('id')) not in seen]
		if self.reranker:
			docs = [{'content': r.get('content',''), **r} for r in results]
			return self.reranker.rerank(query, docs, self.top_n)
		return results[:self.top_n]

	def build_context(self, docs: list) -> str:
		"Format docs into a numbered context string with source metadata."
		parts = []
		for i, d in enumerate(docs, 1):
			meta = d.get('metadata') or ''
			try: src = json.loads(meta).get('source','') if meta else ''
			except: src = meta
			src_str = f' [{src}]' if src else ''
			lvl = d.get('hierarchy_level', 0)
			prefix = '[PARENT] ' if lvl == 1 else ''
			parts.append(f'[{i}]{src_str} {prefix}{d.get("content","")}')
		return '\n\n'.join(parts)

	def _make_rag_tool(self):
		"Returns a plain closure (no @tool decorator) for use as a Lisette tool."
		rag = self
		def search_knowledge_base(query: str) -> str:
			"""Search the local document corpus and return relevant passages."""
			return rag.build_context(rag.retrieve(query))
		return search_knowledge_base

	def query(self, query: str, max_steps: int = 5, stream: bool = False, **kwargs):
		"""Full RAG + agent flow. Calls search_knowledge_base first via Lisette Chat."""
		from lisette import Chat
		rag_tool = self._make_rag_tool()
		all_tools = [rag_tool] + (self.tools or [])
		kw = dict(search=self.search_level) if self.search_level else {}
		chat = Chat(self.llm_model, _RAG_SYSTEM_PROMPT, tools=all_tools, **kw)
		return chat(query, max_steps=max_steps, stream=stream, **kwargs)

	async def aquery(self, query: str, max_steps: int = 5, stream: bool = False, **kwargs):
		"""Async RAG + agent flow using Lisette AsyncChat."""
		from lisette import AsyncChat
		rag_tool = self._make_rag_tool()
		all_tools = [rag_tool] + (self.tools or [])
		kw = dict(search=self.search_level) if self.search_level else {}
		chat = AsyncChat(self.llm_model, _RAG_SYSTEM_PROMPT, tools=all_tools, **kw)
		return await chat(query, max_steps=max_steps, stream=stream, **kwargs)

## GraphRAG (Phase 4 — optional)

`make_explore_relationships_tool` returns a plain function that extracts (subject, predicate, object) triples from retrieved context and stores them in a `graph_triples` SQLite table. Pass it as an extra tool to `LiteRAG(tools=[...])`. No Neo4j, no external services.

In [ ]:
#| export
def _setup_graph_triples(db):
	"Create graph_triples table if it doesn't exist."
	db.execute("""CREATE TABLE IF NOT EXISTS graph_triples (
		id INTEGER PRIMARY KEY,
		subject TEXT NOT NULL,
		predicate TEXT NOT NULL,
		object TEXT NOT NULL,
		source_chunk_id INTEGER
	)""")

def make_explore_relationships_tool(db,            # litesearch Database
								   llm_model: str, # LiteLLM model for triple extraction
):
	"""Returns a plain function for use as a Lisette tool.
	Extracts (subject, predicate, object) triples and stores them in graph_triples."""
	def explore_relationships(chunks_context: str) -> str:
		"""Extract knowledge graph triples from document context and return structured relationships."""
		from lisette import Chat
		_chat = Chat(llm_model,
			'Extract (subject, predicate, object) triples as a JSON array. '
			'Return ONLY valid JSON: [{"subject":"...","predicate":"...","object":"..."}]')
		resp = _chat(chunks_context)
		try:
			triples = json.loads(resp.choices[0].message.content)
			_setup_graph_triples(db)
			db.executemany(
				'INSERT INTO graph_triples (subject,predicate,object) VALUES (?,?,?)',
				[(t['subject'],t['predicate'],t['object']) for t in triples])
			return '\n'.join(
				f"({t['subject']}) --[{t['predicate']}]--> ({t['object']})" for t in triples)
		except Exception as e:
			return f'[graph extraction failed: {e}]'
	return explore_relationships

### Smoke tests (no LLM required)

In [ ]:
# LiteRAG initialisation and retrieve/build_context (no LLM call)
_rag_db = database()
_rag_st = _rag_db.get_store()
_rag_st.insert_all([
	dict(content='The Transformer model uses self-attention mechanisms.'),
	dict(content='BERT is a bidirectional language model pre-trained on masked tokens.'),
	dict(content='Multi-head attention allows attending to information from different positions.'),
])

_rag = LiteRAG(db_path=':memory:', search_level=None)  # search_level=None disables web search
# LiteRAG creates its own in-memory DB, so we inject the test rows directly
_rag.db = _rag_db
_rag.store = _rag_st

_results = _rag.retrieve('attention mechanism')
assert isinstance(_results, list), 'retrieve should return a list'

_ctx = _rag.build_context(_results)
assert isinstance(_ctx, str), 'build_context should return a string'
assert '[1]' in _ctx, 'context should have numbered citations'
print('LiteRAG smoke test passed')
print('Context sample:', _ctx[:200])

In [ ]:
# _make_rag_tool returns a plain function (no decorator) — verify it
_tool = _rag._make_rag_tool()
assert callable(_tool)
assert _tool.__name__ == 'search_knowledge_base'
assert _tool.__doc__ is not None
_tool_result = _tool('self-attention')
assert isinstance(_tool_result, str)
print('_make_rag_tool test passed, tool name:', _tool.__name__)

In [ ]:
# _setup_graph_triples creates the table idempotently
_gdb = database()
_setup_graph_triples(_gdb)
_setup_graph_triples(_gdb)  # must be idempotent
assert 'graph_triples' in _gdb.table_names()
print('graph_triples table test passed')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()